# CSE 30124 - Intro to AI - Homework 02 (10pts.)

* Names:
* NETIDs:

This assignment covers the following topics:

* Decision Trees
* Hidden Markov Models
* Bayesian Reasoning

It will consist of 4 tasks:

| Task ID  | Description                                      | Points |
|----------|--------------------------------------------------|--------|
| 00       | Loading Evidence                |        |
| &nbsp;&nbsp;&nbsp;&nbsp;00-1     | &nbsp;&nbsp;&nbsp;&nbsp;- Load Evidence from Github                   | 0      |
| &nbsp;&nbsp;&nbsp;&nbsp;00-2     | &nbsp;&nbsp;&nbsp;&nbsp;- Add Evidence from Suspect Statements                   | 0      |
| 01       | Decision Trees               |        |
| &nbsp;&nbsp;&nbsp;&nbsp;01-1     | &nbsp;&nbsp;&nbsp;&nbsp;- Feature Engineering                  | 1      |
| &nbsp;&nbsp;&nbsp;&nbsp;01-2     | &nbsp;&nbsp;&nbsp;&nbsp;- Decision Trees                  | 1      |
| &nbsp;&nbsp;&nbsp;&nbsp;01-3     | &nbsp;&nbsp;&nbsp;&nbsp;- Short Answer Questions                  | 2      |
| 02       | Hidden Markov Models               |        |
| &nbsp;&nbsp;&nbsp;&nbsp;02-1     | &nbsp;&nbsp;&nbsp;&nbsp;- Estate Blueprint Creation                 | 1      |
| &nbsp;&nbsp;&nbsp;&nbsp;02-2     | &nbsp;&nbsp;&nbsp;&nbsp;- Viterbi Algorithm                 | 1      |
| &nbsp;&nbsp;&nbsp;&nbsp;02-4     | &nbsp;&nbsp;&nbsp;&nbsp;- Short Answer Questions                 | 2      |
| 03       | Bayesian Reasoning                |        |
| &nbsp;&nbsp;&nbsp;&nbsp;03-1     | &nbsp;&nbsp;&nbsp;&nbsp;- Creation of Prior from Lab 01                 | 1      |
| &nbsp;&nbsp;&nbsp;&nbsp;03-2     | &nbsp;&nbsp;&nbsp;&nbsp;- Bayesian Evidence Combination                 | 1      |
| 04       | Police Report Generation                |        |
| &nbsp;&nbsp;&nbsp;&nbsp;04-1     | &nbsp;&nbsp;&nbsp;&nbsp;- Export Police Report                 | 0      |

Please complete all sections. Some questions may require written answers, while others may involve coding. Be sure to run your code cells to verify your solutions.

### *Story Progression*

Based on the results of your work in `Lab 01`, the police gathered some extra data. After getting in contact with the Theisen-Floyd Estate, the police exported all of the historical evidence to CSV files. They gathered movement and activity patterns of the aliases, plus suspect GPS and witness evidence.

<div class="thumbnail">
    <img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework02/witness.png" class="img-responsive"/>
    <figcaption style="text-align: center"><b>Evidence 1: A witness seeing Mr. Green entering the Pantry!</b></figcaption>
</div>

Additionally, you also manually collected TA statement evidence that needs to be preserved for two of the key suspects (Jack and CLair). By combining the CSV evidence with those TA suspect statements, you can build a complete suspect dataset.

This means that we can then try and map `Suspect -> Alias`.

Thinking back to class, you realize this can be structured as a supervised classification problem. We can compare suspect data against alias data (ground truth) and see if any suspects line up with the aliases!

## Task 00: Loading Evidence
### Task 00-1: Description (0 pts.)

Run the cell below to gather all of the evidence you have for the case so far.

### Task 00-1: Code (0 pts.)

In [ ]:
import os
import pandas as pd

try:
    import google.colab
    REPO_URL = "https://github.com/wtheisen/nd-cse-30124-homeworks.git"

    REPO_PATH = "/content/nd-cse-30124-homeworks"
    L_PATH = "nd-cse-30124-homeworks/evidence/homework02"

    %cd /content/
    !rm -r {REPO_NAME}

    # Clone repo
    if not os.path.exists(REPO_PATH):
        !git clone {REPO_URL}

        # cd into the data folder
        %cd {L_PATH}
        !pwd

except ImportError:
    print(os.getcwd())
    os.chdir("../../evidence/homework02")
    # print(os.getcwd())
finally:
    print("Unable to download repo")
    print("Unable to move to evidence dir")

### Task 00-2: Description (0 pts.)
#### Adding Evidence from Suspect Statements

**Note:** You may need to convert some datatypes again much like you did in `lab00`

Use the CSV evidence files in `evidence/homework02/` to keep data in **DataFrames** for all later tasks.

1. Load `alias_statements.csv` into `alias_df` (2400 rows):

| Column | Description |
|--------|-------------|
| `alias_name` | Alias identity (e.g. "Colonel Mustard") |
| `alias_sample_id` | Integer sample index (0-19 per alias) |
| `time_index` | Observation ordering within sample |
| `time` | Timestamp string (e.g. "19:00 PM") |
| `location` | Room name |
| `activity` | Activity string |

2. Load suspect records into three DataFrames:
   - `suspect_statements_df` (120 rows after adding hardcoded data): columns `suspect_name, statement_index, time, location, activity`
   - `suspect_gps_df` (120 rows): columns `suspect_name, gps_index, time, location, signal_strength`
   - `witness_evidence_df` (merged from `witness_observations.csv` and `witnesses.csv`): columns `suspect_name, witness_name, observation_index, time, location, activity, witness_reliability`

3. Add in the information from the suspect statements you collected in `lab01` for Jack and Clair by building small DataFrames and concatenating them onto `suspect_statements_df`.

4. Define canonical lists: `ALIASES`, `SUSPECTS`, `ROOMS`, `WEAPONS`

---

#### Combining DataFrames with `pd.concat()`

When you have multiple DataFrames with the **same columns** and want to stack them together (append rows), use `pd.concat()`:

```python
# Stack two DataFrames with identical columns on top of each other
combined_df = pd.concat([df_a, df_b], ignore_index=True)
```

- `ignore_index=True` resets the row indices to 0, 1, 2, ... so you don't end up with duplicate indices
- Both DataFrames must have the same column names

* [pd.concat()](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)

---

#### Joining DataFrames with `df.merge()`

When you have two DataFrames that share a **common column** (like a key in a database) and want to combine their information side-by-side, use `df.merge()`:

```python
# Merge observations with witness reliability info, matching on shared columns
witness_evidence_df = witness_obs_df.merge(witnesses_df, on=['suspect_name', 'witness_name'])
```

This is similar to a SQL JOIN -- every row in `witness_obs_df` gets the matching `witness_reliability` value from `witnesses_df` attached to it.

| Method | What it does | When to use |
|--------|-------------|-------------|
| `pd.concat()` | Stacks rows vertically | Combining DataFrames with the **same columns** |
| `df.merge()` | Joins columns horizontally | Combining DataFrames that share a **key column** |

* [df.merge()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)

### Task 00-2: Code (1 pt.)


In [ ]:
### Import necessary libraries ###
import os
import numpy as np
import pandas as pd

from typing import List, Dict, Any
from collections import defaultdict
import pprint

# TODO: Load raw CSV files with pandas (same as lab00)

# TODO: Merge witness observations with witness reliability using df.merge()
#   This joins on the shared columns ['suspect_name', 'witness_name'] so each
#   observation row gets the matching witness_reliability value attached

# TODO: Add Jack and Clair's statements
jack_rows = None
clair_rows = None

# TODO: Use pd.concat() to add Jack and Clair's statements onto the main DataFrame

# Canonical lists
ALIASES = ['Colonel Mustard', 'Miss Scarlet', 'Mr. Green', 'Mrs. Peacock', 'Mrs. White', 'Professor Plum']
SUSPECTS = sorted(suspect_gps_df['suspect_name'].unique().tolist())
ROOMS = ['Study', 'Dining Room', 'Kitchen', 'Pantry', 'Living Room', 'Bathroom']
WEAPONS = ['Poison', 'Knife', 'Gas', 'Rope', 'Bag', 'Firearm']


### *Story Progression*

Looking through the data you have, you realize there's a lot of repeated, overlapping activities and this reminds you of categorical data. You wonder if you could somehow set up a decision tree with the classes being the aliases and then try and match the suspects statements against them!

## Task 01: Decision Trees
### Task 01-1: Description (0 pts.)
#### Feature Engineeering

The first thing we'll need to do is convert our categorical features to numerical ones. Two easy ones would be how much of their time does a person spend in each room and how many activities to they do that could be related to a weapon!

<div class="thumbnail">
    <img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework02/extract_feature.jpg" class="img-responsive"/>
    <figcaption style="text-align: center"><b>Evidence N: Converting categorical features to numerical ones.</b></figcaption>
</div>

In the cell below, finish the code to create a feature engineering function.
---

#### Counting Values with `Series.value_counts()`

When you have a column of categorical data and want to count how many times each value appears, use `value_counts()`:

```python
locations = df['location']
location_counts = locations.value_counts()
# Returns a Series like:
#   Study          8
#   Kitchen        5
#   Dining Room    3
#   ...

# Access a specific count (returns 0 if the value isn't found)
location_counts.get('Study', 0)  # Returns: 8
location_counts.get('Pantry', 0)  # Returns: 0
```

* [Series.value_counts()](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)

---

#### Mapping Values with `Series.map()`

When you want to convert every value in a column using a dictionary lookup, use `Series.map()`:

```python
weapon_map = {'Opening a safe': 'Firearm', 'Taking her dog for a walk': 'Rope'}

# Map each activity to its weapon (unmapped activities become NaN)
weapons_used = df['activity'].map(weapon_map)
# Returns a Series like:
#   0    Firearm
#   1        NaN    <- activity wasn't in the dictionary
#   2       Rope

# Use .dropna() to remove NaN values
weapons_used.dropna()  # Only keeps the successfully mapped values
```

**Note:** `.map()` is much cleaner than writing a for-loop with `if activity in dict` checks!

* [Series.map()](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html)

### Task 01-1: Code (1 pt.)


In [ ]:
# Note: There should be 3 activities per weapon
activity_weapon_mapping = {
    'Inflating balloons': 'Gas',
    'Ripping wippets': 'Gas',
    #TODO: Add a third Gas activity

    'Whittling a stick': 'Knife',
    'Peeling an orange': 'Knife',
    'Opening letters': 'Knife',

    'Tampering with drinks': 'Poison',
    'Rinsing a glass carefully': 'Poison',
    #TODO: Add a third Poison activity

    'Tying decorative knots': 'Rope',
    'Raising a painting': 'Rope',
    'Taking her dog for a walk': 'Rope',

    'Inspecting a family heirloom': 'Firearm',
    'Opening a safe': 'Firearm',
    'Talking about skeet shooting': 'Firearm',

    'Bragging about their Birkin': 'Bag',
    'Complaining about TSA going through their purse': 'Bag',
    #TODO: Add a third Bag activity
}

def extract_features(statements_df: pd.DataFrame) -> dict:
    '''
    Function converts the categorical data we have into numerical data for the decision tree.
    Accepts a DataFrame with 'location' and 'activity' columns.
    '''

    rooms = ['Study', 'Dining Room', 'Kitchen', 'Pantry', 'Living Room', 'Bathroom']
    weapons = ['Poison', 'Knife', 'Gas', 'Rope', 'Bag', 'Firearm']

    features = {}

    total_time = len(statements_df)

    # TODO: Count room visits using value_counts() instead of a manual loop
    #   value_counts() returns a Series with locations as index and counts as values

    # TODO: Map activities to weapons using Series.map() instead of manual lookups
    #   .map(dict) replaces each activity with its weapon (or NaN if not in the dict)
    #   .dropna() removes NaN values, set() gives us unique weapons

    return features


### Task 01-2: Description (0 pts.)
#### Preparing Data for Classification

Now that we have our function to create numerical features, lets prepare the data for our training data (aliases) and what we want to test (suspect data).

To set up for our decision tree, lets store our data in two DataFrames:

- `alias_features_df`: one row per alias sample, with columns for `alias_name`, `alias_sample_id`, plus all feature columns
- `suspect_features_df`: one row per suspect, with columns for `suspect_name` plus all feature columns

---

#### Splitting and Applying with `df.groupby().apply()`

In `lab01` we used `df.apply(func, axis=1)` to run a function on **each row**. But what if we want to run a function on **each group of rows**? For example, each alias has 20 sample statements -- we want to extract features from each sample separately.

`groupby()` splits a DataFrame into groups based on one or more columns, then `apply()` runs a function on each group:

```python
# Group by alias_name and sample_id, then extract features from each group
alias_features_df = alias_df.groupby(['alias_name', 'alias_sample_id']) \
    .apply(lambda g: pd.Series(extract_features(g)), include_groups=False) \
    .reset_index()
```

Breaking this down step-by-step:

| Step | What it does |
|------|-------------|
| `.groupby(['alias_name', 'alias_sample_id'])` | Splits the DataFrame into one sub-DataFrame per unique (alias, sample) pair |
| `.apply(lambda g: ...)` | Calls the lambda on each sub-DataFrame `g` |
| `extract_features(g)` | Our function from Task 01-1, returns a dict of features |
| `pd.Series(...)` | Converts the dict into a pandas Series (one row of data) |
| `include_groups=False` | Don't pass the grouping columns into the function |
| `.reset_index()` | Converts the group keys back into regular columns |

* [df.groupby()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)
* [GroupBy.apply()](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.GroupBy.apply.html)

### Task 01-2: Code (0 pts.)


In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import export_text

alias_features_df = alias_df.groupby(['alias_name', 'alias_sample_id'], sort=False) \
    .apply(lambda g: pd.Series(extract_features(g)), include_groups=False).reset_index()

suspect_features_df = suspect_statements_df.groupby('suspect_name', sort=False) \
    .apply(lambda g: pd.Series(extract_features(g)), include_groups=False).reset_index()


### Task 01-3: Description (0 pts.)
#### Decision Tree Classifier

Now that we have our two datasets, lets fit our decision tree onto the alias data and then try and classify our suspect data and see what we get!

### Task 01-3: Code (0 pts.)

In [ ]:
# Define feature columns (same rooms and weapons from extract_features)
feature_cols = [f'time_in_{room}' for room in ROOMS] + [f'suggests_{weapon}' for weapon in WEAPONS]

# TODO: Define training data from alias features
#   .values converts DataFrame columns to a numpy array (same as lab01)


# TODO: Train Decision Tree Classifier
#   use arguments criterion='entropy', random_state=42, max_depth=4, min_samples_split=4


# TODO: Define test data and run decision tree on test data


# Print Results
print("\nProbability of Each Suspect Matching Each Alias:\n")
print(f"{clf.classes_}")

aliases = list(clf.classes_)

for suspect_name, probs in zip(suspect_features_df['suspect_name'], probabilities):
    formatted_probs = [f"{v:.4f}" for v in probs]
    print(f'{suspect_name}: {formatted_probs}')

# Store decision tree results as DataFrame (index=suspect_name, columns=alias names)
dt_results_df = pd.DataFrame(probabilities, index=suspect_features_df['suspect_name'].values, columns=aliases)


### Task 01-3: Expected Output (1 pt.)

```
Probability of Each Suspect Matching Each Alias:

['Colonel Mustard' 'Miss Scarlet' 'Mr. Green' 'Mrs. Peacock' 'Mrs. White' 'Professor Plum']
Cesar Cervera: ['0.0000', '0.0000', '0.8636', '0.0000', '0.0455', '0.0909']
Olivia Zino: ['0.0000', '0.0000', '0.0000', '1.0000', '0.0000', '0.0000']
Sophia Noonan: ['0.0000', '0.9412', '0.0000', '0.0000', '0.0000', '0.0588']
Tom Lohman: ['0.5000', '0.0000', '0.0000', '0.5000', '0.0000', '0.0000']
Jack Mangione: ['0.0000', '0.0000', '0.0000', '0.0000', '0.0000', '1.0000']
Clair Green: ['0.0000', '0.2000', '0.0000', '0.0000', '0.8000', '0.0000']
```


### Task 01-4: Short Answer Questions (2 pts.)

1. Why are we using criterion='entropy'? Explain what "entropy" means in this context and what the tree is maximizing at each split.
   * [ANSWER]

2. Do we need to scale or normalize our features for a decision tree? Why or why not?
   * [ANSWER]


### *Story Progression*

Nice! We've got a decision tree that can classify the suspects into the aliases! However lets try and get some more evidence before we make a final decision. The decision tree makes use of the time spent in rooms, but it doesn't consider the paths the suspects take between the rooms. Suddenly inspiration strikes! You recall learning about Hidden Markov Models in class and think that maybe you could use them to get some more evidence!

The police have acquired a blueprint of the estate and based on visitor data the likelihood that a person moves between rooms.

<div class="thumbnail">
    <img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework02/estate.png" class="img-responsive"/>
    <figcaption style="text-align: center"><b>Evidence N: Blueprint of the Estate, handmade!</b></figcaption>
</div>

**Note:** I am not an artist don't laugh!

## Task 02: Hidden Markov Model
### Task 02-1: Description (0 pts.)
#### Creating Blueprint of Estate
    
1. Create a blueprint dictionary that gives the transition probabilities between rooms in the estate. Name the dictionary `blueprint`, with the format as follows:
```
"rooms": {
    "ROOM NAME": {"id": ROOM ID, "adjacent_to": ["NEIGHBOR 1", "NEIGHBOR 2"]}
    },
"transition_probabilities": {
    ROOM ID: {
        NEIGHBOR 1 ID: 0.4,
        NEIGHBOR 2 ID: 0.4,
    }
}
```

   * The transition probabilities are given below:
      0. Study:
         * Dining Room: 0.3
         * Living Room: 0.3
         * Study: 0.4
      1. Dining Room:
         * Kitchen: 0.25
         * Study: 0.25
         * Living Room: 0.25
         * Dining Room: 0.25
      2. Kitchen:
         * Dining Room: 0.4
         * Pantry: 0.4
         * Kitchen: 0.2
      3. Pantry:
         * Kitchen: 0.8
         * Pantry: 0.2
      4. Living Room:
         * Study: 0.25
         * Dining Room: 0.25
         * Bathroom: 0.25
         * Living Room: 0.25
      5. Bathroom:
         * Living Room: 0.7
         * Bathroom: 0.3

3. Using a hidden markov model, find the most likely path for a suspect given the GPS data and witness statements

4. Compare the suspect's alleged path from their statement you collected to the most likely path found by the HMM
    
### Task 02-1: Code (1 pt.)

In [ ]:
import numpy as np
from typing import List, Dict, Tuple
from collections import defaultdict

# TODO: Initialize blueprint.json
blueprint = {
    "rooms": { },
    "transition_probabilities": { }
}

locations = blueprint['rooms']
n_states = len(locations)

id_to_room = {info['id']: room for room, info in locations.items()}

initial = np.ones(n_states) / n_states

### Task 02-2: Description (0 pts.)
#### Creating Emission Probabilities

For each timestep, we need to figure out the probability that a suspect was in each room based on the GPS and witness evidence. We'll do this by filtering our DataFrames to just the current time, then summing up the evidence.

---

#### Filtering DataFrames by Time

In `lab01` we saw how to filter DataFrames using boolean indexing (e.g., `df[df['col'] == value]`). We'll use the same pattern here to get just the rows matching a specific timestamp:

```python
# Get all GPS readings at a specific time for this suspect
gps_at_time = suspect_gps[suspect_gps['time'] == '19:00 PM']

# Check if we got any results
if not gps_at_time.empty:
    row = gps_at_time.iloc[0]  # Get the first (and usually only) matching row
    signal = row['signal_strength']
```

* [df.empty](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.empty.html)

---

#### Iterating Over Rows with `df.iterrows()`

When you need to loop over each row of a DataFrame and access its values, use `iterrows()`:

```python
# Loop over each witness observation at this time
for _, obs in witness_at_time.iterrows():
    location = obs['location']
    reliability = obs['witness_reliability']
```

**Note:** The `_` variable is the row index -- we use `_` because we don't need it. Each `obs` is a pandas Series representing one row, and you access columns like a dictionary.

* [df.iterrows()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iterrows.html)

### Task 02-2: Code (0 pts.)


In [ ]:
def weight_room_matrix(time: str, suspect_gps: pd.DataFrame, suspect_witness_evidence: pd.DataFrame) -> np.ndarray:
    """
    Construct a single emission probability vector for the HMM at the specified time.
    GPS signal strengths and witness reliabilities are interpreted as pseudo-counts
    (independent evidence contributions) for the room in question. We add a tiny
    Dirichlet-style prior so the downstream log operations stay finite, then
    normalise to obtain a valid distribution summing to one.
    """

    epsilon = 1e-6  # Numerical floor to keep probabilities strictly positive.
    emission = np.full(n_states, epsilon, dtype=float)  # Prior mass keeps logs finite.
    evidence = np.zeros(n_states, dtype=float)

    gps_at_time = suspect_gps[suspect_gps['time'] == time]
    if not gps_at_time.empty:
        row = gps_at_time.iloc[0]  # Get the first matching row
        gps_strength = float(np.clip(row.get('signal_strength', 0.0), 0.0, 1.0))
        gps_loc = row.get('location')
        if gps_loc in blueprint['rooms']:
            gps_idx = blueprint['rooms'][gps_loc]['id']
            evidence[gps_idx] += gps_strength

    witness_at_time = suspect_witness_evidence[suspect_witness_evidence['time'] == time]
    for _, obs in witness_at_time.iterrows():
        reliability = float(np.clip(obs.get('witness_reliability', 0.0), 0.0, 1.0))
        if reliability == 0.0:
            continue
        location = obs.get('location')
        if location in blueprint['rooms']:
            room_idx = blueprint['rooms'][location]['id']
            evidence[room_idx] += reliability

    if evidence.sum() == 0.0:
        emission += np.ones(n_states, dtype=float)
    else:
        emission += evidence

    emission /= emission.sum()
    return emission


### Task 02-3: Description (0 pts.)
#### The Viterbi Algorithm

The reconstruct the most likely paths that each suspect took through the estate, based on the evidence, we can use the Viterbi algorithm! In the cell below, implement the viterbi algorithm. Remember that the psuedo-code for this was shown during lecture!

### Task 02-3: Code (1 pt.)

In [ ]:
def viterbi(suspect_gps: pd.DataFrame, suspect_witness_evidence: pd.DataFrame, timestamps: List[str]) -> Tuple[List[str], float]:
    """
    Use Viterbi algorithm to find most likely sequence of true locations.
    Also returns the probability of the sequence.
    """
    total_timesteps = len(timestamps)
    num_rooms = len(blueprint["rooms"])

    V = np.zeros((total_timesteps, num_rooms))
    backptr = np.full((total_timesteps, num_rooms), -1, dtype=int)

    # Create emission matrices for all timestamps
    emissions = []
    for t in timestamps:
        emission_t = weight_room_matrix(t, suspect_gps, suspect_witness_evidence)
        emissions.append(emission_t)

    # Initialize first timestep
    V[0] = initial * emissions[0]

    for time_step in range(1, total_timesteps):
        # TODO: Implement the Viterbi algorithm
        pass
        

    # Backtrack to reconstruct path
    path = []
    current = np.argmax(V[-1])
    path.append(current)

    for time_step in range(total_timesteps - 1, 0, -1):
        # TODO: Reconstruct path
        pass

    path.reverse()
    room_path = [id_to_room[i] for i in path]

    sequence_prob = np.max(V[-1])

    return room_path, sequence_prob


### Task 02-4: Description (0 pts.)
#### HMM Analysis

Now that we have the HMM set up, and our viterbi algorithm, run the cell below to check how much each suspect's stated path agrees with their most likely path according to the HMM. The more they are lying about their locations the more disagreements they'll *probably* have with the HMM path!

---

#### Useful DataFrame Methods for This Task

A couple of handy pandas methods you'll see in the cell below:

```python
# Get unique values from a column (returns a numpy array)
suspect_statements_df['suspect_name'].unique()
# Returns: ['Cesar Cervera', 'Olivia Zino', 'Sophia Noonan', ...]

# Sort a DataFrame by a column
df.sort_values('time')  # Returns a new DataFrame sorted by the 'time' column

# Convert a column to a Python list
df['location'].tolist()  # Returns: ['Study', 'Kitchen', 'Study', ...]
```

* [Series.unique()](https://pandas.pydata.org/docs/reference/api/pandas.Series.unique.html)
* [df.sort_values()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html)
* [Series.tolist()](https://pandas.pydata.org/docs/reference/api/pandas.Series.tolist.html)

### Task 02-4: Code (0 pts.)


In [ ]:
# Test the corrected implementation
print("HMM Path Analysis:")
print("=" * 50)

hmm_records = []

for suspect_name in suspect_statements_df['suspect_name'].unique():
    # Filter each DataFrame to just this suspect using boolean indexing
    s_statements = suspect_statements_df[suspect_statements_df['suspect_name'] == suspect_name]
    s_gps = suspect_gps_df[suspect_gps_df['suspect_name'] == suspect_name]
    s_witness = witness_evidence_df[witness_evidence_df['suspect_name'] == suspect_name]

    # Get sorted timestamps from this suspect's statements
    timestamps = sorted(s_statements['time'].tolist())
    
    true_path, path_prob = viterbi(s_gps, s_witness, timestamps)
    # Sort statements by time and extract the stated locations as a list
    stated_path = s_statements.sort_values('time')['location'].tolist()
    
    matches = sum(1 for true, stated in zip(true_path, stated_path) if true == stated)
    agreement = matches / len(true_path)

    print(f"{suspect_name}:")
    print(f'\tAgreement of Statement Path with HMM Path: {agreement:.2f}, Number of Contradictions: {len(true_path) - matches}')

    hmm_records.append({
        'suspect_name': suspect_name,
        'path_probability': path_prob,
        'statement_agreement': agreement,
        'num_contradictions': len(true_path) - matches
    })

# Build results DataFrame with suspect_name as the index (for easy .loc[] access later)
hmm_results_df = pd.DataFrame(hmm_records).set_index('suspect_name')


### Task 02-4: Expected Output (1 pt.)

```
HMM Path Analysis:
==================================================
Cesar Cervera:
	Agreement of Statement Path with HMM Path: 0.25, Number of Contradictions: 15
Olivia Zino:
	Agreement of Statement Path with HMM Path: 0.80, Number of Contradictions: 4
Sophia Noonan:
	Agreement of Statement Path with HMM Path: 0.75, Number of Contradictions: 5
Tom Lohman:
	Agreement of Statement Path with HMM Path: 0.65, Number of Contradictions: 7
Jack Mangione:
	Agreement of Statement Path with HMM Path: 0.65, Number of Contradictions: 7
Clair Green:
	Agreement of Statement Path with HMM Path: 0.80, Number of Contradictions: 4
```


### Task 02-5: Short Answer Questions (2 pts.)

1. What are the key components of a Hidden Markov Model? What do the transition and emission probabilities represent in this problem?
   * [ANSWER]

2. How do we create the emissions matrix?
   * [ANSWER]

### *Story Progression*

Looks like we have a lot of lying suspects, but one seems to be lying more than the others. The evidence is beginning to stack up against Mr. Cervera! *Nice*, now we have two pieces of evidence that we can use to match the suspects to the aliases! If only there were some way to combine them. As you finish your drink you think back to class and remember that you can use Bayesian reasoning to combine the evidence! With your knowledge of bayesian reasoning, hopefully we be able to figure out who was who!

## Task 03: Bayesian Reasoning
### Task 03-1: Description (0 pts.)
#### Creating the Prior from Lab 01

In Bayesian reasoning, a **prior** represents our initial belief about something *before* we see new evidence. In `Lab 01`, you computed the average lying probability for each suspect from their polygraph test. We can use this as **evidence** to update a uniform prior via Bayes' rule.

Since we know that **Mr. Green is the liar**, suspects with higher polygraph lying scores should have a higher posterior probability of being Mr. Green.

We'll construct a prior probability distribution over aliases for each suspect using a **Bayesian update**:

1. Start with a **uniform prior** over aliases: $P(\text{alias}) = \frac{1}{6}$
2. Define a **likelihood model** for the polygraph score given each alias. The key idea: we treat the polygraph lying score (`lying_score`) as a *measurement* whose value depends on whether the suspect is actually a liar:
   - If the alias is Mr. Green (the liar), a high lying score is *expected* — so we use the score directly as the likelihood: $P(s \mid \text{Mr. Green}) = s$, where $s$ = `lying_score`
   - If the alias is truthful, a high lying score is *surprising* — so we flip it: $P(s \mid \text{other}) = 1 - s$
   - For example, a suspect with a 0.8 lying score gets 0.8 likelihood for Mr. Green but only 0.2 for any truthful alias — the evidence strongly favors the liar hypothesis
3. Apply **Bayes' rule**: $P(\text{alias} \mid \text{polygraph}) \propto P(\text{alias}) \times P(s \mid \text{alias})$
4. **Normalize** so probabilities sum to 1

---

Load the lying probabilities from the `Lab 01` results CSV, then finish the `create_prior` function.

### Task 03-1: Code (0 pts.)


In [ ]:
def create_prior(lab01_df: pd.DataFrame, aliases: list) -> pd.DataFrame:
    """
    Create prior probability distributions by performing a Bayesian update
    on a uniform prior using polygraph lying probabilities as evidence.

    P(alias | polygraph) ~ P(alias) * P(polygraph | alias)

    Likelihood model:
      P(lying_score | liar_alias) = lying_score    (liar expected to score high)
      P(lying_score | other alias) = 1 - lying_score  (truthful expected to score low)
    """
    n = len(aliases)
    base_prior = 1.0 / n

    records = []
    for _, row in lab01_df.iterrows():
        suspect_name = row['suspect_name']
        lying_prob = row['total_lying_prob']

        # TODO: Update uniform priors

        # TODO: Normalize posteriors

    # Build a DataFrame with suspect_name as index and alias names as columns
    return pd.DataFrame(records).set_index('suspect_name')

lab01_df = pd.read_csv('../../evidence/homework02/lab01_lying_probabilities.csv')

print("Lab 01 Lying Probabilities:")
for _, row in lab01_df.iterrows():
    print(f"  {row['suspect_name']}: {row['total_lying_prob']:.3f}")

priors_df = create_prior(lab01_df, aliases)

# Print priors
print("\nPrior Distributions:")
for suspect_name in priors_df.index:
    print(f"{suspect_name}:")
    for alias in priors_df.columns:
        # Use .loc[row, col] to access a specific value (same as lab00)
        print(f"  {alias}: {priors_df.loc[suspect_name, alias]:.3f}")


### Task 03-1: Expected Output (1 pt.)
```
Lab 01 Lying Probabilities:
  Olivia Zino: 0.505
  Jack Mangione: 0.784
  Tom Lohman: 0.783
  Cesar Cervera: 0.515
  Sophia Noonan: 0.575
  Clair Green: 0.779

Prior Distributions:
Olivia Zino:
  Professor Plum: 0.166
  Colonel Mustard: 0.166
  Mr. Green: 0.169
  Mrs. White: 0.166
  Miss Scarlet: 0.166
  Mrs. Peacock: 0.166
Jack Mangione:
  Professor Plum: 0.116
  Colonel Mustard: 0.116
  Mr. Green: 0.421
  Mrs. White: 0.116
  Miss Scarlet: 0.116
  Mrs. Peacock: 0.116
Tom Lohman:
  Professor Plum: 0.116
  Colonel Mustard: 0.116
  Mr. Green: 0.419
  Mrs. White: 0.116
  Miss Scarlet: 0.116
  Mrs. Peacock: 0.116
Cesar Cervera:
  Professor Plum: 0.165
  Colonel Mustard: 0.165
  Mr. Green: 0.175
  Mrs. White: 0.165
  Miss Scarlet: 0.165
  Mrs. Peacock: 0.165
Sophia Noonan:
  Professor Plum: 0.157
  Colonel Mustard: 0.157
  Mr. Green: 0.213
  Mrs. White: 0.157
  Miss Scarlet: 0.157
  Mrs. Peacock: 0.157
Clair Green:
  Professor Plum: 0.117
  Colonel Mustard: 0.117
  Mr. Green: 0.413
  Mrs. White: 0.117
  Miss Scarlet: 0.117
  Mrs. Peacock: 0.117
```

### Task 03-2: Description (0 pts.)
#### Bayesian Evidence Combination

Now we combine our **prior** (Lab 01 polygraph) with two **likelihoods** (evidence from this homework):

$$P(\text{Alias} \mid \text{Evidence}) \propto \underbrace{P(\text{Alias})}_{\text{Prior}} \times \underbrace{P(\text{Activities} \mid \text{Alias})}_{\text{Decision Tree}} \times \underbrace{P(\text{Agreement} \mid \text{Alias})}_{\text{HMM}}$$

Where:
- **P(Alias)** is the prior from Task 03-1
- **P(Activities | Alias)** comes from the decision tree probabilities (Task 01)
- **P(Agreement | Alias)** is another likelihood, this time using the HMM path agreement from Task 02. The reasoning follows the same pattern as the prior's likelihood model — we have a measurement (path agreement) whose expected value depends on whether the suspect is a liar:
  - If the alias is **Mr. Green** (the liar): a liar would fabricate their statement, so their stated path should *disagree* with the HMM-inferred true path. High agreement is surprising for a liar, so: `truth_factor = 1 - path_agreement`
  - If the alias is **truthful**: a truthful suspect would report where they actually were, so their stated path should *agree* with the HMM path. High agreement is expected, so: `truth_factor = path_agreement`
  - For example, Cesar Cervera has only 0.25 path agreement — giving him a `1 - 0.25 = 0.75` likelihood for Mr. Green but only `0.25` for any truthful alias

After computing the unnormalized posteriors, **normalize** so probabilities sum to 1 for each suspect.

### Task 03-2: Code (0 pts.)

In [ ]:
def combine_evidence_bayesian(priors_df: pd.DataFrame, dt_results_df: pd.DataFrame, hmm_results_df: pd.DataFrame, aliases: list) -> pd.DataFrame:
    """
    Combine prior, decision tree, and HMM evidence using Bayes' theorem.

    P(Alias|Evidence) ~ P(Alias) * P(Activities|Alias) * P(Agreement|Truthful)

    Args:
        priors_df: DataFrame (index=suspect_name, columns=alias names)
        dt_results_df: DataFrame (index=suspect_name, columns=alias names)
        hmm_results_df: DataFrame (index=suspect_name, columns include 'statement_agreement')
        aliases: list of alias names

    Returns:
        DataFrame mapping suspect_name (index) -> alias posterior probabilities (columns)
    """
    records = []

    for suspect_name in dt_results_df.index:
        posterior = {}

        # TODO: Retrieve decision tree probabilities using .loc[suspect_name]

        # TODO: Retrieve path agreement score using .loc[suspect_name, column]

        # TODO: Retrieve prior for this suspect

        # TODO: For each alias, combine prior * DT likelihood * HMM likelihood

        # TODO: Normalize posteriors

        normalized['suspect_name'] = suspect_name
        records.append(normalized)

    return pd.DataFrame(records).set_index('suspect_name')

final_results_df = combine_evidence_bayesian(priors_df, dt_results_df, hmm_results_df, aliases)

# Print Results
for suspect_name in final_results_df.index:
    print(f"{suspect_name}:")
    for alias in final_results_df.columns:
        print(f"  {alias}: {final_results_df.loc[suspect_name, alias]:.3f}")


### Task 03-2: Expected Results (1 pt.)

```
Cesar Cervera:
  Colonel Mustard: 0.000
  Miss Scarlet: 0.000
  Mr. Green: 0.953
  Mrs. Peacock: 0.000
  Mrs. White: 0.016
  Professor Plum: 0.031
Olivia Zino:
  Colonel Mustard: 0.000
  Miss Scarlet: 0.000
  Mr. Green: 0.000
  Mrs. Peacock: 1.000
  Mrs. White: 0.000
  Professor Plum: 0.000
Sophia Noonan:
  Colonel Mustard: 0.000
  Miss Scarlet: 0.941
  Mr. Green: 0.000
  Mrs. Peacock: 0.000
  Mrs. White: 0.000
  Professor Plum: 0.059
Tom Lohman:
  Colonel Mustard: 0.500
  Miss Scarlet: 0.000
  Mr. Green: 0.000
  Mrs. Peacock: 0.500
  Mrs. White: 0.000
  Professor Plum: 0.000
Jack Mangione:
  Colonel Mustard: 0.000
  Miss Scarlet: 0.000
  Mr. Green: 0.000
  Mrs. Peacock: 0.000
  Mrs. White: 0.000
  Professor Plum: 1.000
Clair Green:
  Colonel Mustard: 0.000
  Miss Scarlet: 0.200
  Mr. Green: 0.000
  Mrs. Peacock: 0.000
  Mrs. White: 0.800
  Professor Plum: 0.000
```

### *Final Story Progression*

<div class="thumbnail">
    <img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework02/suspect.png" class="img-responsive"/>
    <figcaption style="text-align: center"><b>Evidence 4: The prime suspect, Cesar!</b></figcaption>
</div>

We did it, it seems extremely likely that Cesar Cervera is Mr. Green! You quickly call up Detective Caulfield and explain your findings. He laughs at you. It seems that a purely statistical analysis isn't going to be enough for the authorities to make a warrant. You pour yourself another drink and start brainstorming other ways to find more conclusive evidence.

## Task 04: Police Report Generation
### Task 04: Description (0 pts.)

Edit and run the cell below to generate your report to submit to the authorities on Canvas.

### Task 04: Code (0 pts.)


In [ ]:
import os, json

ASS_PATH = "nd-cse-30124-homeworks/homeworks"
ASS = "homework02"

GENERATE = False

if GENERATE:
    try:
        from google.colab import _message, files

        # where you WANT it to live (repo folder)
        repo_ipynb_path = f"/content/{ASS_PATH}/{ASS}/{ASS}.ipynb"

        # grab current notebook contents from the UI
        nb = _message.blocking_request("get_ipynb", timeout_sec=1)["ipynb"]

        # write it into the repo folder as a real file
        os.makedirs(os.path.dirname(repo_ipynb_path), exist_ok=True)
        with open(repo_ipynb_path, "w", encoding="utf-8") as f:
            json.dump(nb, f)

        # convert + download html
        !jupyter nbconvert --to html "{repo_ipynb_path}"
        files.download(repo_ipynb_path.replace(".ipynb", ".html"))
    except:
        import subprocess

        nb_fp = os.getcwd() + f'/{ASS}.ipynb'
        print(os.getcwd())

        subprocess.run(["jupyter", "nbconvert", "--to", "html", nb_fp], check=True)
    finally:
        print('[WARNING]: Unable to export notebook as .html')